# Production manager service testbench

Prerequisites: source ROS 2 Jazzy and the workspace overlay, use a Jupyter kernel from `/opt/venv`, and start the production manager in local mode. Configure and activate the node before running the service cells. This notebook does not change lifecycle state.

In [ ]:
import ctypes, glob
for so in sorted(glob.glob('/workspaces/autofactory/install/production_manager/lib/libproduction_manager__*.so')):
    ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)

In [ ]:
import rclpy
from production_manager.srv import GetProductionData, ReportProduction, SetOperation

rclpy.init()
node = rclpy.create_node('production_manager_testbench')
TARGET_NODE = '/production_manager'

In [ ]:
OPERATION_ID = 'dab06081-0e24-4583-9f33-06d3c26b69f7'
SERIAL_NUMBER = 'ABCDEF'
STATION_ID = 'bc10eaad-47b8-46d5-b66f-62a41a2b9b91'
PRODUCT_NUMBER = 'ABC123/1'
PRODUCT_REVISION = '1.2'
ASSEMBLY_ID = '6384f49c-a0d5-4d8d-bdcf-a45c6f02d20d'
OPERATION_SUCCEEDED = True

In [ ]:
def call(service_type, service_name, request, timeout=5.0):
    client = node.create_client(service_type, f'{TARGET_NODE}/{service_name}')
    try:
        if not client.wait_for_service(timeout_sec=timeout):
            raise TimeoutError(f'Service {service_name} was not available within {timeout}s')
        future = client.call_async(request)
        rclpy.spin_until_future_complete(node, future, timeout_sec=timeout)
        if not future.done():
            raise TimeoutError(f'Service call {service_name} did not complete within {timeout}s')
        return future.result()
    finally:
        node.destroy_client(client)

In [ ]:
request = SetOperation.Request()
request.operation_id = OPERATION_ID
request.serial_number = SERIAL_NUMBER
request.station_id = STATION_ID
set_operation_response = call(SetOperation, 'set_operation', request)
print(set_operation_response)

In [ ]:
request = GetProductionData.Request()
request.product_number = PRODUCT_NUMBER
request.product_revision = PRODUCT_REVISION
request.serial_number = SERIAL_NUMBER
get_production_data_response = call(GetProductionData, 'get_production_data', request)
print(get_production_data_response)

In [ ]:
request = ReportProduction.Request()
request.assembly_id = ASSEMBLY_ID
request.operation_id = OPERATION_ID
request.success = OPERATION_SUCCEEDED
report_production_response = call(ReportProduction, 'report_production', request)
print(report_production_response)

In [ ]:
node.destroy_node()
rclpy.shutdown()